In [ ]:
from pathlib import Path

import duckdb

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
CSV = (
    ROOT
    / "dados/raw/prestacao_contas/2026/prestacao_contas_candidatos_2026/receitas_candidatos_doador_originario_2026_BRASIL.csv"
)
PARQUET = ROOT / "dados/processed/receitas_candidatos_doador_originario.parquet"
con = duckdb.connect()
con.execute("LOAD encodings")
con.sql(
    f"CREATE OR REPLACE VIEW dados_raw AS SELECT * FROM read_csv('{CSV.as_posix()}', delim=';', header=true, encoding='cp1252', all_varchar=true, ignore_errors=true)"
)
con.sql(f"COPY dados_raw TO '{PARQUET.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
con.sql(f"CREATE VIEW dados AS SELECT * FROM '{PARQUET.as_posix()}'")

In [ ]:
con.sql("DESCRIBE dados").df()

In [ ]:
con.sql("SUMMARIZE dados").df()

In [ ]:
con.sql("EXPLAIN SELECT * FROM dados").df()

In [ ]:
con.sql("SHOW ALL TABLES")